# Turbine Generation

In [0]:
# %pip install pymc
import pymc as pm
import numpy as np
import pandas as pd
import arviz as az

In [0]:
# Small Dataset: 20 observations of power output (kW)
data = np.array([15.2, 16.1, 14.8, 15.5, 15.0, 14.9, 15.8, 16.2, 15.1, 14.7, 
                 15.3, 15.4, 14.6, 15.9, 15.2, 15.0, 15.1, 15.5, 14.8, 15.2])

display(data)

In [0]:
with pm.Model() as model:
    # Priors: We expect the mean (mu) to be around 15 based on historical data
    mu = pm.Normal("mu", mu=15, sigma=2)
    # Standard deviation (sigma) must be positive
    sigma = pm.HalfNormal("sigma", sigma=1)
    
    # Likelihood: Defining how the data is distributed
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=data)
    
    # Inference: Drawing 2000 samples from the posterior
    trace = pm.sample(2000, return_inferencedata=True, progressbar=False)

In [0]:
# Display the summary of the posterior distribution
summary = az.summary(trace)
display(summary)

# Plot the distribution
az.plot_posterior(trace)

# Clinical Trials

In [0]:
import pymc as pm
import numpy as np
import arviz as az

In [0]:

# Scenario: A trial with only 12 patients. 9 showed positive response.
# We have a Prior belief from Phase 1 that the drug has a ~60% success rate.
trials = 12
successes = 9

In [0]:
with pm.Model() as clinical_trial:
    # Prior: Beta(6, 4) represents a prior belief of 60% success
    theta = pm.Beta("success_rate", alpha=6, beta=4)
    
    # Likelihood: The actual observed data from the 12 patients
    obs = pm.Binomial("obs", n=trials, p=theta, observed=successes)
    
    # MCMC Sampling to find the 'Posterior' (updated belief)
    trace = pm.sample(2000, return_inferencedata=True, target_accept=0.95)

In [0]:

# Calculate the probability that the success rate is > 70%
post_samples = trace.posterior["success_rate"].values
prob_high_success = (post_samples > 0.70).mean()

print(f"Probability the drug success rate is > 70%: {prob_high_success:.2%}")
az.plot_posterior(trace, ref_val=0.70)